In [ ]:
import csv
import numpy as np
import pandas as pd
import sys, os
import re
import ast
import datetime as dt
from tqdm import tqdm
from collections import defaultdict, Counter

import warnings

warnings.filterwarnings("ignore")

from sklearn.preprocessing import MultiLabelBinarizer

In [ ]:
df = pd.read_csv('D:/mimic-iv-3.1/mimiciv/3.1/icu/icustays.csv.gz')
df = df[df.los>=1]
df.shape

(74829, 8)

In [3]:
ori = ['PROC_221214', 'PROC_221216', 'PROC_221217', 'PROC_221223', 'PROC_221255', 'PROC_223253', 'PROC_224263', 'PROC_224264', 'PROC_224267', 'PROC_224270', 'PROC_224385', 'PROC_225400', 'PROC_225401', 'PROC_225402', 'PROC_225432', 'PROC_225441', 'PROC_225444', 'PROC_225451', 'PROC_225454', 'PROC_225462', 'PROC_225752', 'PROC_225792', 'PROC_225794', 'PROC_225802', 'PROC_225814', 'PROC_225817', 'PROC_225966', 'PROC_227194', 'PROC_229351', 'MEDS_221289', 'MEDS_221468', 'MEDS_221555', 'MEDS_221653', 'MEDS_221662', 'MEDS_221744', 'MEDS_221749', 'MEDS_221794', 'MEDS_221906', 'MEDS_221986', 'MEDS_222042', 'MEDS_222056', 'MEDS_222168', 'MEDS_222315', 'MEDS_223258', 'MEDS_225152', 'MEDS_225168', 'MEDS_225170', 'MEDS_229630', 'OUT_226571', 'OUT_226579', 'OUT_226627']
ids = []
for i in ori:
    ids.append(int(i.split('_')[1]))

In [4]:
d_items = pd.read_csv('D:/mimic-iv-3.1/mimiciv/3.1/icu/d_items.csv.gz')

In [5]:
proc = pd.read_csv('D:/mimic-iv-3.1/mimiciv/3.1/icu/procedureevents.csv.gz')

In [6]:
proc=proc[['stay_id','itemid','starttime', 'endtime', 'value', 'ordercategoryname']]
proc=proc[proc['stay_id'].isin(df['stay_id'].unique())]

In [7]:
proc.head(2)

,stay_id,itemid,starttime,endtime,value,ordercategoryname
3,37081114,224275,2150-11-02 20:00:00,2150-11-03 13:00:00,1020.0,Peripheral Lines
4,37081114,224277,2150-11-02 20:38:00,2150-11-03 11:59:00,921.0,Peripheral Lines


In [8]:
procs = [225401, 225437, 225444, 225451, 225454, 225722, 225723, 225724,
225725, 225726, 225727, 225728, 225729, 225730, 225731, 225732,
225733, 225734, 225735, 225736, 225768, 225814, 225816, 225817,
225818, 226131, 227726,225399,225400,225401,225427,225430,225433,225434,225439,225440,225441,225442,225445,225448,225449,225450,225464,225464,225466,225467,
225472,225475,225479,225752,225792,225794,225802,225805,225966,225967,226237,227194,227712,227713,228715,229351,
224385,221214,221216,221217,221223,223253,224264,224268,224270,224272,224273,225402]
procs = pd.unique(procs)
len(procs)

72

In [9]:
proc=proc[proc['itemid'].isin(procs)]
proc=proc.dropna(subset=['value'])

In [12]:
proc.head(2)

,stay_id,itemid,starttime,endtime,value,ordercategoryname
6,37081114,225402,2150-11-05 12:00:00,2150-11-05 12:01:00,1.0,Procedures
7,37081114,225401,2150-11-05 17:21:00,2150-11-05 17:22:00,1.0,Procedures


In [13]:
proc = pd.merge(proc,df[['stay_id', 'intime', 'outtime']],on='stay_id',how='left')

In [14]:
proc['intime'] = pd.to_datetime(proc['intime'])
proc['outtime'] = pd.to_datetime(proc['outtime'])
proc['starttime'] = pd.to_datetime(proc['starttime'])
proc['endtime'] = pd.to_datetime(proc['endtime'])

In [16]:
within_criteria = proc['starttime'].between(proc['intime'],proc['outtime'])
within = proc[within_criteria]

In [17]:
within['event_time_from_admit'] = within['starttime'] - within['intime']
within['start_time']=within['event_time_from_admit'].dt.total_seconds() / 3600
within['los'] = (within['outtime'] - within['intime']).dt.total_seconds() / 3600
within=within[within['start_time']>=0]
within.head(2)

,stay_id,itemid,starttime,endtime,value,ordercategoryname,intime,outtime,event_time_from_admit,start_time,los
0,37081114,225402,2150-11-05 12:00:00,2150-11-05 12:01:00,1.0,Procedures,2150-11-02 19:37:00,2150-11-06 17:03:17,2 days 16:23:00,64.383333,93.438056
1,37081114,225401,2150-11-05 17:21:00,2150-11-05 17:22:00,1.0,Procedures,2150-11-02 19:37:00,2150-11-06 17:03:17,2 days 21:44:00,69.733333,93.438056


In [18]:
within_72 = within[within['start_time']<=72]
within_72.shape

(235457, 11)

In [ ]:
def is_numeric(value):
    try:
        float(value)  
        return True
    except (ValueError, TypeError):
        return False

within_72['is_numeric'] = within_72['value'].apply(is_numeric)

result = within_72.groupby('itemid')['is_numeric'].all().reset_index()
result.rename(columns={'is_numeric': 'is_fully_numeric'}, inplace=True)

num_col = result[result.is_fully_numeric==True].itemid.values
str_col = result[result.is_fully_numeric==False].itemid.values

In [ ]:
def compute_outlier_imputation(arr, upper_percentile, lower_percentile):

    arr = pd.to_numeric(arr, errors='coerce')

    upper_bound = np.percentile(arr, upper_percentile)
    lower_bound = np.percentile(arr, lower_percentile)
    
    arr[arr < lower_bound] = lower_bound
    arr[arr > upper_bound] = upper_bound

    return arr

def outlier_imputation(data, id_attribute, value_attribute, num_col, upper_percentile=98, lower_percentile=2):

    grouped = data.groupby(id_attribute)

    for id_number, group in grouped:
        if id_number not in num_col:
            continue
        
        index = group.index
        values = group[value_attribute].values

        imputed_values = compute_outlier_imputation(values, upper_percentile, lower_percentile)
        
        data.loc[index, value_attribute] = imputed_values

    data = data.dropna(subset=[value_attribute])

    return data

In [23]:
within_72 = outlier_imputation(within_72, 'itemid', 'value', num_col, upper_percentile=98, lower_percentile=2)

In [ ]:
final_chart=pd.DataFrame()
t=0

for i in tqdm(np.arange(0, 72, 0.25)):  
    sub_chart = within_72[(within_72['start_time'] >= i) & (within_72['start_time'] < i + 0.25)]
    
    sub_chart_num = sub_chart[sub_chart['itemid'].isin(num_col)].groupby(['stay_id', 'itemid']).agg({'value': np.nanmean}).reset_index()

    sub_chart_str = sub_chart[sub_chart['itemid'].isin(str_col)].groupby(['stay_id', 'itemid']).agg({'value': lambda x: list(x)}).reset_index()

    sub_chart = pd.concat([sub_chart_num, sub_chart_str], ignore_index=True)

    sub_chart['start_time'] = t

    if final_chart.empty:
        final_chart = sub_chart
    else:
        final_chart = final_chart.append(sub_chart)

    t += 1

100%|████████████████████████████████████████████████████████████████████████████████| 288/288 [00:12<00:00, 22.77it/s]


In [28]:
cate_01 = []
for it in final_chart.itemid.unique():
    if len(final_chart[final_chart.itemid==it].value.unique())>2:
        print(it,d_items[d_items.itemid==it].label.values)
        print(final_chart[final_chart.itemid==it].value.value_counts())
    else:
        cate_01.append(it)

225752 ['Arterial Line']
18378.000000    608
2.944444        600
1260.000000      46
1140.000000      45
1200.000000      41
               ... 
17896.000000      1
17040.000000      1
3882.000000       1
6410.000000       1
8455.000000       1
Name: value, Length: 9493, dtype: int64
225792 ['Invasive Ventilation']
26368.98    547
107.00      546
210.00       62
240.00       58
300.00       57
           ... 
7904.00       1
21598.00      1
7280.00       1
4900.00       1
7215.00       1
Name: value, Length: 9004, dtype: int64
229351 ['Foley Catheter']
2.513806        327
22165.760000    327
1200.000000      19
1440.000000      17
900.000000       17
               ... 
7553.000000       1
194.000000        1
5817.000000       1
8702.000000       1
9419.000000       1
Name: value, Length: 7511, dtype: int64
224264 ['PICC Line']
27499.880000    186
4.965583        185
1586.000000       9
1441.000000       8
1650.000000       7
               ... 
4339.000000       1
838.000000        1


In [31]:
process_final_chart = final_chart.copy()

In [32]:
process_final_chart.shape

(233445, 4)

In [ ]:
los=288
def create_Dict(chart):
    dataDic = {}
    feat = chart['itemid'].unique()  
    
    for hid in tqdm(chart.stay_id.unique()):  
        df2 = chart[chart['stay_id'] == hid]  
        
        if df2.shape[0] == 0:
            print(f'空：{hid}')
            val = pd.DataFrame(np.zeros([los, len(feat)]), columns=feat)  
            val = val.fillna(0)  

        else:
            
            val = df2.pivot(index='start_time', columns='itemid', values='value')
            
            add_indices = pd.Index(range(los)).difference(val.index)
            add_df = pd.DataFrame(index=add_indices, columns=val.columns).fillna(0)  
            val = pd.concat([val, add_df])  
            val = val.sort_index()  
            
           
            feat_df = pd.DataFrame(columns=list(set(feat) - set(val.columns)))  
            val = pd.concat([val, feat_df], axis=1)  
            
            val = val[feat]  
            val = val.fillna(0)  

        val.to_csv(f'MIMIC_IV/Final_Proc/{hid}.csv', index=False)

In [36]:
create_Dict(process_final_chart)

100%|████████████████████████████████████████████████████████████████████████████| 57475/57475 [28:18<00:00, 33.84it/s]


In [ ]:
def add_dynamic_process(dynamic, cate_01, str_col):
    dynamic.replace(0, np.nan, inplace=True)
    
    dynamic.dropna(axis=1, how='all', inplace=True)
    
    dynamic.columns = [
        f"PROC_Cate_{col}" if int(col) in cate_01 else 
        f"PROC_Str_{col}" if int(col) in str_col else 
        f"PROC_{col}" 
        for col in dynamic.columns
    ]
    
    for col in dynamic.columns:
        if col.startswith('PROC_Cate_'):
            dynamic[col] = dynamic[col].fillna(0)
        else:
            dynamic[col] = dynamic[col].ffill().bfill()
    
    return dynamic

In [ ]:
save_path = '/MIMIC_IV/Final_Proc_2/'

for i in tqdm(os.listdir('/MIMIC_IV/Final_Proc/')):
    try:
        file_path = f'/MIMIC_IV/Final_Proc/{i}'
        dynamic = pd.read_csv(file_path)
        dynamic = add_dynamic_process(dynamic, cate_01, str_col)
        
        save_file_path = os.path.join(save_path, i)
        dynamic.to_csv(save_file_path, index=False)
    
    except Exception as e:
        print(f"Error processing file {i}: {e}")

100%|████████████████████████████████████████████████████████████████████████████| 57475/57475 [47:59<00:00, 19.96it/s]


In [40]:
cols_count = Counter()

In [ ]:
def median_ts_process(ts):
    ts.loc[:, (ts == 0).all(axis=0)] = np.nan  
    
    processed_values = {}
    for col in ts.columns:
        if col.startswith('PROC_Cate_'):  
            processed_values[col] = ts[col].max()
        elif col.startswith('PROC_Str_'):  
            non_zero_values = ts[col][ts[col] != '0']
            if not non_zero_values.empty:
                processed_values[col] = non_zero_values.mode().iloc[0]
            else:
                processed_values[col] = 0
        else:
            processed_values[col] = ts[col].median()
    
    result = pd.DataFrame([processed_values])
    
    return result

In [ ]:
all_ts = pd.DataFrame()

for i in tqdm(os.listdir('/MIMIC_IV/Final_Proc_2/')):
    file_path = f'/MIMIC_IV/Final_Proc_2/{i}'
    ts = pd.read_csv(file_path)
    
    ts = median_ts_process(ts)
    
    cols_count.update(ts.columns)
    
    ts['ICUSTAY_ID'] = i.split('.')[0]
    
    all_ts = pd.concat([all_ts, ts], ignore_index=True)
    
all_ts = all_ts.dropna(axis=1, how='all')

100%|████████████████████████████████████████████████████████████████████████████| 57475/57475 [21:27<00:00, 44.65it/s]


In [46]:
delete = ['PROC_Cate_226237']

In [47]:
keep_all_ts = all_ts.drop(columns=delete, errors='ignore')

In [48]:
keep_all_ts.shape

(57475, 53)

In [ ]:
str_cols = []
cate_cols = []
c_cols = []
icuid = []

for col in keep_all_ts.columns:
    if col.startswith('PROC_Str_'):
        str_cols.append(col)  
    elif col.startswith('PROC_Cate_'):
        cate_cols.append(col)
    elif col=='ICUSTAY_ID':
        icuid.append(col)
    else:
        c_cols.append(col)


In [51]:
feat = icuid + c_cols+cate_cols+str_cols
len(feat)

53

In [52]:
keep_all_ts = keep_all_ts[feat]

In [ ]:
keep_all_ts.to_csv('/MIMIC_IV/All_Proc_mean_max_mode_nan.csv',index=False)